### This was necessary because the eigenvectors were saved in a custom text format, 
### and we need to read them back into numpy arrays for analysis. 

In [ ]:
import numpy as np

def read_eigenvectors(filename="eigenvectors.txt"):
    eigenvalues = []
    eigenvectors = []

    with open(filename, "r") as f:
        lines = f.readlines()

    current_vec = []

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # header
        if line.startswith("# eigenvectors"):
            continue

        # eigenvalue line
        if line.startswith("# eigenvalue"):
            # save previous vector
            if current_vec:
                eigenvectors.append(np.array(current_vec, dtype=np.float64))
                current_vec = []

            # parse eigenvalue
            val = float(line.split("=")[1])
            eigenvalues.append(val)

        else:
            # vector entries
            nums = [float(x) for x in line.split()]
            current_vec.extend(nums)

    # last vector
    if current_vec:
        eigenvectors.append(np.array(current_vec, dtype=np.float64))

    eigenvalues = np.array(eigenvalues)
    eigenvectors = np.array(eigenvectors)

    return eigenvalues, eigenvectors


# usage
evals, evecs = read_eigenvectors()

print("eigenvalues shape:", evals.shape)
print("eigenvectors shape:", evecs.shape)

print("\nlowest eigenvalues:")
print(evals[:10])

print("\nfirst vector norm:")
print(np.linalg.norm(evecs[0]))

eigenvalues shape: (30,)
eigenvectors shape: (30, 130873600)

lowest eigenvalues:
[-11.86883557 -11.86883557 -11.86883557 -11.82490062 -11.82490062
 -11.82490062 -11.82490062 -11.82490062 -11.82490062 -11.77617927]

first vector norm:
1.0000000000000646


In [2]:
np.savetxt("eigenvalues_4x4_7_7_pbc_30.txt", evals[:10])
np.savetxt("eigenvectors_4x4_7_7_pbc_30.txt", evecs[:10])

In [6]:
#
# save eigenvalues
#
evals.astype(np.float32).tofile(
    "eigenvalues_4x4_7_7_pbc_30_f32.bin"
)

#
# save eigenvectors as float32
#
evecs.transpose().astype(np.float32).tofile(
    "eigenvectors_4x4_7_7_pbc_30_f32.bin"
)

In [5]:
evecs.transpose().shape

(130873600, 30)

In [7]:
np.linalg.norm(evecs[0,:])

np.float64(1.0000000000000646)

In [10]:
evecs[0,2]

np.float64(6.762122010400979e-10)

In [11]:
import netket as nk
import netket.experimental as nkx
import numpy as np
from flax import nnx
from netket.utils import struct
from netket.sampler.rules import ExchangeRule
from netket.hilbert import SpinOrbitalFermions
import os

pbc = [True, True]

L_x = 4
L_y = 4
N_up = N_down = 7

hilb = nk.hilbert.SpinOrbitalFermions(
    n_orbitals= L_x * L_y, s=1/2, n_fermions_per_spin=(N_up, N_down)
)

t = 1
U = 8

def c(site, sz):
    return nk.operator.fermion.destroy(hilb, site, sz=sz)


def cdag(site, sz):
    return nk.operator.fermion.create(hilb, site, sz=sz)


def nc(site, sz):
    return nk.operator.fermion.number(hilb, site, sz=sz)

g = nk.graph.Grid(extent=[L_x, L_y], pbc=pbc)

up = +1
down = -1
ham = 0.0

for sz in (up, down):
    for u, v in g.edges():
        ham += -t * cdag(u, sz) @ c(v, sz) - t * cdag(v, sz) @ c(u, sz)
for u in g.nodes():
    ham += U * nc(u, up) @ nc(u, down)


H8 = nkx.operator.ParticleNumberAndSpinConservingFermioperator2nd.from_fermionoperator2nd(
    ham
)

∣NK⟩ Tip: uv is a replacement for pip which helps you follow good software practices.

In [19]:
import jax
import jax.numpy as jnp
from jax import lax
jax.config.update("jax_enable_x64", False)


alpha = 2
n_layers = 1
n_hidden_fermions = 14

PF_size = 2 * hilb.n_orbitals + N_up + N_down + n_hidden_fermions * 2
scale = 0.1

lr = 0.02
diag_shift = 0.01

n_samples = 1024
n_steps = 4000
chunk_size = None

n_hidden = alpha * (hilb.n_orbitals * 2)


rngs = nnx.Rngs(jax.random.PRNGKey(0))
# ma8 = LogHiddenFermion(hilb, eps=0.2, rngs=rngs, hidden_units=n_hidden, PF_size=PF_size, n_layers=n_layers, scale=scale, n_hidden_fermions=n_hidden_fermions) #, activation=nnx.tanh)
sa8 = nk.sampler.MetropolisFermionHop(hilb, graph=g,
                                      n_chains=16,
                                      sweep_size=4*hilb.n_orbitals)
# vs8 = nk.vqs.MCState(sa8, ma8, n_samples=n_samples, chunk_size=chunk_size)

print("hidden units:", n_hidden)
print("PF size:", PF_size)
print("n_layers:", n_layers)
# print("WF parameters:", count_params(ma8))

# optE = nk.optimizer.Sgd(learning_rate=lr)  
# gs8 = nk.driver.VMC_SR(H8, optE, variational_state=vs8, diag_shift=diag_shift, mode="real")

n_samples = 1024
rngs = nnx.Rngs(jax.random.PRNGKey(0))

vs_target2 = nk.vqs.MCState(
    sampler=sa8,
    model=nk.models.LogStateVector(hilb, param_dtype=jnp.complex128),
    n_samples=n_samples,
    variables={"params": {"logstate": jnp.log(evecs[0,:].astype(jnp.complex128)).squeeze()}},
)

vs_target2.expect(H8)


hidden units: 64
PF size: 74
n_layers: 1


/home/detlef/anaconda3/envs/NetKet/lib/python3.12/site-packages/jax/_src/nn/initializers.py:92: UserWarning: Explicitly requested dtype complex128 requested in ones is not available, and will be truncated to dtype complex64. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return jnp.ones(shape, dtype, out_sharding=out_sharding)
/home/detlef/anaconda3/envs/NetKet/lib/python3.12/site-packages/jax/_src/interpreters/mlir.py:1172: UserWarning: A large amount of constants were captured during lowering (4.19GB total). If this is intentional, disable this warning by setting JAX_CAPTURED_CONSTANTS_WARN_BYTES=-1. To obtain a report of where these constants were encountered, set JAX_CAPTURED_CONSTANTS_REPORT_FRAMES=-1.
  warnings.warn(message)
/home/detlef/anaconda3/envs/NetKet/lib/python3.12/site-packages/jax/_src/nn/initializers.py:92: UserWarning: Explicitly reques

21.47-0.00j ± 0.36 [σ²=9.6e+01, R̂=1.083]

In [21]:
import numpy as np
import netket as nk
from itertools import combinations

Ns = 16
Nup = 7
Ndown = 7

#
# reproduce C++ basis
#

def make_basis(Ns, Nf):
    out = []

    for occ in combinations(range(Ns), Nf):
        s = 0
        for i in occ:
            s |= (1 << i)
        out.append(s)

    return out

up_basis = make_basis(Ns, Nup)
down_basis = make_basis(Ns, Ndown)

dim = len(up_basis) * len(down_basis)

#
# NetKet expects shape:
# (n_states, 2*Ns)
#

states = np.zeros((dim, 2 * Ns), dtype=np.int8)

idx = 0

for iu, up in enumerate(up_basis):
    for idn, dn in enumerate(down_basis):

        #
        # first all UP
        #

        for site in range(Ns):
            states[idx, site] = (up >> site) & 1

        #
        # then all DOWN
        #

        for site in range(Ns):
            states[idx, Ns + site] = (dn >> site) & 1

        idx += 1

print(states.shape)

#
# NetKet Hilbert
#

hilb = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=Ns,
    s=1/2,
    n_fermions_per_spin=(Nup, Ndown),
)

numbers = hilb.states_to_numbers(states)

print(numbers[:20])

#
# compare ordering
#

d = np.diff(numbers)

print("all consecutive =", np.all(d == 1))

if np.all(d == 1):
    print("same ordering")
else:
    print("different ordering")

#
# permutation:
# cpp_index -> netket_index
#

perm = np.argsort(numbers)

#
# verify
#

print(
    np.all(numbers[perm] == np.arange(dim))
)

np.save("cpp_to_netket_perm.npy", perm)

(130873600, 32)


/home/detlef/anaconda3/envs/NetKet/lib/python3.12/site-packages/jax/_src/interpreters/mlir.py:1172: UserWarning: A large amount of constants were captured during lowering (4.19GB total). If this is intentional, disable this warning by setting JAX_CAPTURED_CONSTANTS_WARN_BYTES=-1. To obtain a report of where these constants were encountered, set JAX_CAPTURED_CONSTANTS_REPORT_FRAMES=-1.
  warnings.warn(message)


[130873599 130873598 130873597 130873596 130873595 130873594 130873593
 130873592 130873591 130873590 130873589 130873588 130873587 130873586
 130873585 130873584 130873583 130873582 130873581 130873580]
all consecutive = False
different ordering
True


In [22]:
perm = np.argsort(numbers)

In [23]:
evecs_netket = evecs[:, perm]

In [24]:
import jax
import jax.numpy as jnp
from jax import lax
jax.config.update("jax_enable_x64", False)


alpha = 2
n_layers = 1
n_hidden_fermions = 14

PF_size = 2 * hilb.n_orbitals + N_up + N_down + n_hidden_fermions * 2
scale = 0.1

lr = 0.02
diag_shift = 0.01

n_samples = 1024
n_steps = 4000
chunk_size = None

n_hidden = alpha * (hilb.n_orbitals * 2)


rngs = nnx.Rngs(jax.random.PRNGKey(0))
# ma8 = LogHiddenFermion(hilb, eps=0.2, rngs=rngs, hidden_units=n_hidden, PF_size=PF_size, n_layers=n_layers, scale=scale, n_hidden_fermions=n_hidden_fermions) #, activation=nnx.tanh)
sa8 = nk.sampler.MetropolisFermionHop(hilb, graph=g,
                                      n_chains=16,
                                      sweep_size=4*hilb.n_orbitals)
# vs8 = nk.vqs.MCState(sa8, ma8, n_samples=n_samples, chunk_size=chunk_size)

print("hidden units:", n_hidden)
print("PF size:", PF_size)
print("n_layers:", n_layers)
# print("WF parameters:", count_params(ma8))

# optE = nk.optimizer.Sgd(learning_rate=lr)  
# gs8 = nk.driver.VMC_SR(H8, optE, variational_state=vs8, diag_shift=diag_shift, mode="real")

n_samples = 1024
rngs = nnx.Rngs(jax.random.PRNGKey(0))

vs_target2 = nk.vqs.MCState(
    sampler=sa8,
    model=nk.models.LogStateVector(hilb, param_dtype=jnp.complex128),
    n_samples=n_samples,
    variables={"params": {"logstate": jnp.log(evecs_netket[0,:].astype(jnp.complex128)).squeeze()}},
)

vs_target2.expect(H8)


hidden units: 64
PF size: 74
n_layers: 1


-11.86883450-0.00000001j ± 0.00000012 [σ²=1.5e-11, R̂=1.013]

In [25]:
#
# save eigenvectors as float32
#
evecs_netket.transpose().astype(np.float32).tofile(
    "eigenvectors_4x4_7_7_pbc_30_f32.bin"
)